In [14]:
import pandas as pd
import numpy as np
from sklearn import linear_model, metrics
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
import random
import os

In [15]:
SEED = 42 
# Python RNG 
random.seed(SEED) 
# NumPy RNG 
np.random.seed(SEED) 
# Optional: full determinism for sklearn parallel algorithms 
os.environ["PYTHONHASHSEED"] = str(SEED) 
os.environ["OMP_NUM_THREADS"] = "1" 
os.environ["MKL_NUM_THREADS"] = "1"

In [16]:
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv')
y_test = pd.read_csv('../data/y_test.csv')

In [17]:
baseline = np.ones(len(y_train))*np.average(y_train)
mse_baseline = metrics.mean_squared_error(y_train, baseline)

clf = linear_model.LinearRegression()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
mse_st = metrics.mean_squared_error(y_test, y_pred)

print(mse_baseline / mse_st)

5.883119489837891


In [18]:
class Seeded:
    def __init__(self, estimator):
        self.estimator = estimator

    def __call__(self, *args, **kwargs):
        # Case 1: estimator is a class (e.g., RandomForestRegressor)
        if hasattr(self.estimator, "get_params"):
            # Instantiate a temporary object to inspect parameters
            params = self.estimator().get_params()
            if "random_state" in params and "random_state" not in kwargs:
                kwargs["random_state"] = SEED
            return self.estimator(*args, **kwargs)

        # Case 2: estimator is a function (e.g., train_test_split)
        if hasattr(self.estimator, "__code__"):
            if "random_state" in self.estimator.__code__.co_varnames and "random_state" not in kwargs:
                kwargs["random_state"] = SEED
            return self.estimator(*args, **kwargs)

        # Fallback
        return self.estimator(*args, **kwargs)

In [ ]:
models = {
    "Linear (double split)": linear_model.LinearRegression(),
    "Ridge": Seeded(linear_model.Ridge)(alpha=0.1),
    "Lasso": Seeded(linear_model.Lasso)(alpha=0.01),
    "kNN": KNeighborsRegressor(n_neighbors=50, weights="distance"),
    "ElasticNet": Seeded(linear_model.ElasticNet)(alpha=0.1, l1_ratio=0.5),
    "RandomForestRegressor": Seeded(RandomForestRegressor)(n_estimators=100, max_depth=20)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train.values.ravel())
    
    y_test_pred = model.predict(X_test)
    mse_test = metrics.mean_squared_error(y_test, y_test_pred)
    
    results[name] = mse_test
    print(f"{name}: MSE test = {mse_test:.4f}")

best_model_name = min(results, key=results.get)
print(f"\nNajlepszy model: {best_model_name}")

y_pred = models[best_model_name].predict(X_test)
mse_best = metrics.mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse_best)
mae  = metrics.mean_absolute_error(y_test, y_pred)
r2   = metrics.r2_score(y_test, y_pred)
mape = metrics.mean_absolute_percentage_error(y_test, y_pred)

print("\n=== Ewaluacja najlepszego modelu (test) ===")
print(f"MSE:   {mse_best:.2f}")
print(f"RMSE:  {rmse:.2f}")
print(f"MAE:   {mae:.2f}")
print(f"R2:    {r2:.4f}")
print(f"MAPE:  {mape*100:.2f}%")

Linear (double split): MSE test = 153656546.8121
Ridge: MSE test = 153652226.3429


c:\Users\magda\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.968e+12, tolerance: 7.585e+09
  model = cd_fast.enet_coordinate_descent(


Lasso: MSE test = 153638946.2534
kNN: MSE test = 93839167.0980
ElasticNet: MSE test = 166778545.3436
RandomForestRegressor: MSE test = 32950391.1385

Najlepszy model: RandomForestRegressor

=== Ewaluacja najlepszego modelu (test) ===
MSE:   32950391.14
RMSE:  5740.24
MAE:   3612.63
R2:    0.9633
MAPE:  10.41%


In [20]:
### Log-ification of database

results_log = {}
for name, model in models.items():

    model.fit(X_train, np.log1p(y_train.values.ravel()))
    

    y_test_pred_log = model.predict(X_test)
    y_test_pred = np.expm1(y_test_pred_log)
    
    mse_test = metrics.mean_squared_error(y_test, y_test_pred)
    results_log[name] = mse_test
    print(f"{name}: MSE test (log-trained) = {mse_test:.4f}")


best_log_name = min(results_log, key=results_log.get)
print(f"\nNajlepszy model (skala log.): {best_log_name}")


best_model = models[best_log_name]
best_model.fit(X_train, np.log1p(y_train.values.ravel()))

y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)

mse_log = metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse_log)
mae  = metrics.mean_absolute_error(y_test, y_pred)
r2   = metrics.r2_score(y_test, y_pred)
mape = metrics.mean_absolute_percentage_error(y_test, y_pred)

print("\n=== Ewaluacja najlepszego modelu logarytmicznego (test) ===")
print(f"MSE:   {mse_log:.2f}")
print(f"RMSE:  {rmse:.2f}")
print(f"MAE:   {mae:.2f}")
print(f"R2:    {r2:.4f}")
print(f"MAPE:  {mape*100:.2f}%")

Linear (double split): MSE test (log-trained) = 143242948.8004
Ridge: MSE test (log-trained) = 143234619.0010
Lasso: MSE test (log-trained) = 179006124.8525
kNN: MSE test (log-trained) = 97571323.1206
ElasticNet: MSE test (log-trained) = 248311707.7443
RandomForestRegressor: MSE test (log-trained) = 33185062.4445

Najlepszy model (skala log.): RandomForestRegressor

=== Ewaluacja najlepszego modelu logarytmicznego (test) ===
MSE:   33185062.44
RMSE:  5760.65
MAE:   3614.23
R2:    0.9631
MAPE:  9.97%
